# Код дообучения модели Qwen для Google Colab

Установка библиотек

In [ ]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes
!pip install unsloth_zoo

Загрузка базовой модели Qwen 2.5 3B

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048 # Максимальная длина контекста
dtype = None # Автовыбор (float16 для T4)
load_in_4bit = True # 4-битное квантование для экономии VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Настройка LoRA адаптеров
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Ранг LoRA
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Загрузка и форматирование датасета

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Подготавливаем токенизатор под шаблон диалогов Qwen 2.5
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

# Функция приведения сообщений к текстовому формату Qwen
def format_prompts(examples):
    texts = [tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = False) for messages in examples["messages"]]
    return { "text" : texts }

# Загружаем JSONL
dataset = load_dataset("json", data_files={"train": "dataset.jsonl"}, split="train")
dataset = dataset.map(format_prompts, batched = True)

Запуск процесса обучения

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Эффективный батч = 8
        warmup_steps = 10,
        num_train_epochs = 3, # 3 эпохи для 500 примеров
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

Экспорт модели в формат GGUF

In [ ]:
# Сохранение в 4-битный формат Q4_K_M для Ollama
model.save_pretrained_gguf("qwen2.5_ddd_model", tokenizer, quantization_method = "q4_k_m")

Выгрузка на диск

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp -r /content/qwen2.5_ddd_model_gguf /content/drive/MyDrive/